In [158]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

cuda


In [2]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(len(text))
print(text[:200])

232309
﻿  DOROTHY AND THE WIZARD IN OZ

  BY

  L. FRANK BAUM

  AUTHOR OF THE WIZARD OF OZ, THE LAND OF OZ, OZMA OF OZ, ETC.

  ILLUSTRATED BY JOHN R. NEILL

  BOOKS OF WONDER WILLIAM MORROW & CO., INC. NEW


In [3]:
chars = sorted(set(text))
print(chars)
print(len(chars))
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']
81


Tokenizer contains an encoder and a decoder

what encoder does it covnerts each element of array to integer

In [4]:
# basic char level tokenizer
# takes a char and covnerts into a integer

string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

encoded_hello = encode('hello')
print(encoded_hello)
decoded_hello = decode(encoded_hello)
print(decoded_hello)

[61, 58, 65, 65, 68]
hello


Tokenizer contains an encoder and a decoder

what encoder does it covnerts each element of array to integer

In [5]:
# basic char level tokenizer
# takes a char and covnerts into a integer

string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

encoded_hello = encode('hello')
print(encoded_hello)
decoded_hello = decode(encoded_hello)
print(decoded_hello)


[61, 58, 65, 65, 68]
hello


In [6]:
# converting to tensor type
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([80,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])


In [7]:
# # train and val data split
# n = int(0.8*len(data))
# train_data = data[:n]
# val_data = data[n:]

In [8]:
# train_data

In [9]:
# x = train_data[:block_size]
# y = train_data [1:block_size+1]

# for t in range(block_size):
#     context = x[:t+1]
#     target = y[t]
#     print('when  input is ', context, 'target is ', target)

In [10]:
# train and val data split
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
# print(x.shape)
print(x)
print('targets:')
print(y)

inputs:
tensor([[ 1, 62, 67,  1, 54,  1, 76, 62],
        [60,  1, 54,  1, 59, 58, 76,  1],
        [73, 72,  1, 68, 59,  1, 54,  1],
        [75, 58,  1, 78, 68, 74,  1, 55]], device='cuda:0')
targets:
tensor([[62, 67,  1, 54,  1, 76, 62, 67],
        [ 1, 54,  1, 59, 58, 76,  1, 68],
        [72,  1, 68, 59,  1, 54,  1, 55],
        [58,  1, 78, 68, 74,  1, 55, 58]], device='cuda:0')


In [162]:
# this decorator makes sure pytorch does not uses gratients at all in here, that will reduce computation & memory usage
# so it's overall better for performance, and becoz we are just reporting a loss, we dont really need to do any optimizing or gradient conputation here
# for any outside funtion which is used for evalutation and where model is being passed and we dont want to use gradients then use this decorator
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [11]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


WDC[u-z﻿Qp&.JKM-O.Jd3H)[3]XB);MiU

c0
No*V7xiKBc[M0Xd6svhvz
pg5
zVs&RW3S'AsI7﻿dkd;nZOl 6sT3]Z,-EGGQ0m0mNO"﻿Tp:3,lQ Wd0TU1rE,4wW9-o
NCQ1KKd61h6E&[E1;E*alm
ZVQ;BSYUg INI:P]X7i2"mem!5Y(
zmx!AcNq8kQqqA.B8E&.7PwQvnHF;F*!itQhxuW6-jX:ZR*1KMVMFW1vZITEd6P&.n'kX'B5sF;k-CDp07
g926-N;MF;ZV7d8b,!dE8QHFK&&e_khm M R8U1KM)_oJbe:?lS!a7RKdZR*xphwi1ZI2k?tdZ:5&XBTFmaD4:t?﻿[WfLI5X'VZ;1"B4)Olm-?vz4w-8u77Rxma﻿La﻿[kfYW[NF(b"'1cY8x!Owmq2"_o*wNKb)rbrXytZ"rJmLrsRiBT*X4wHC[yR6odKuj﻿TlV﻿'FU1&X'
W1K7i)zmLD T4!BqlnC;f)LsJBS[a


In [159]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 2.406, val loss: 2.518
step: 250, train loss: 2.422, val loss: 2.492
step: 500, train loss: 2.464, val loss: 2.486
step: 750, train loss: 2.445, val loss: 2.511
2.1928749084472656


In [161]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)



HE st I mpot,"ime thqugrontesor mooutuland ayskeerofo us veask anount the mprely linome st, in t ca I witid s thast s id athalllery hed be berotr fomany I ably." "

" basorease hethe, fe we ns."Dimubbl wou thed, at t outhe way hind wl." toonneee fllthe hey we r;  acad m
thark y."I hind m and

esatemf tan IANon othald MOzin mus Wersoforngay ent jon mas th oy.
"blo wonomithinowang ce  he we I walisere sor tofum. prackancknle
"PThas ss RR the t sod alin y f astinthere, Drtrewaurearid
wat but.


Th


need to familiarize audience with optimizers (AdamW, Adam, SGD, MSE…) no need to jump into the formulas, just what the optimizer does for us and some of the differences/similarities between them

Mean Squared Error (MSE): MSE is a common loss function used in regression problems, where the goal is to predict a continuous output. It measures the average squared difference between the predicted and actual values, and is often used to train neural networks for regression tasks.
Gradient Descent (GD): is an optimization algorithm used to minimize the loss function of a machine learning model. The loss function measures how well the model is able to predict the target variable based on the input features. The idea of GD is to iteratively adjust the model parameters in the direction of the steepest descent of the loss function
Momentum: Momentum is an extension of SGD that adds a "momentum" term to the parameter updates. This term helps smooth out the updates and allows the optimizer to continue moving in the right direction, even if the gradient changes direction or varies in magnitude. Momentum is particularly useful for training deep neural networks.
RMSprop: RMSprop is an optimization algorithm that uses a moving average of the squared gradient to adapt the learning rate of each parameter. This helps to avoid oscillations in the parameter updates and can improve convergence in some cases.
Adam: Adam is a popular optimization algorithm that combines the ideas of momentum and RMSprop. It uses a moving average of both the gradient and its squared value to adapt the learning rate of each parameter. Adam is often used as a default optimizer for deep learning models.
AdamW: AdamW is a modification of the Adam optimizer that adds weight decay to the parameter updates. This helps to regularize the model and can improve generalization performance. We will be using the AdamW optimizer as it best suits the properties of the model we will train in this video.
find more optimizers and details at torch.optim